# Modelagem Supervisionada

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

from model_selection import get_best_model
from model_tuning import tune_xgboost

from evaluation_plots import plot_confusion_matrix, plot_roc_curve, plot_feature_importances


pd.set_option('display.max_columns', None)

In [2]:
#importando dados
df = pd.read_pickle('../data/curated/features.pkl')
df.head()

,IS_DELAYED,AIRPLANE_WAS_DELAYED,ORIGIN_AIRPORT_DELAY_MOMENTUM,DESTINATION_AIRPORT_DELAY_MOMENTUM,ORIGIN_AIRPORT_FLIGHTS_ON_THE_SAME_WINDOW,MONTH,SCHEDULED_DEPARTURE_HOUR,IS_WEEKEND,IS_HOLIDAY,HAUL_TYPE_SHORT,HAUL_TYPE_MEDIUM,HAUL_TYPE_LONG
0,1,-0.547400,-0.898713,-0.904391,-0.887294,-0.338099,-1.588269,1.716522,-0.161727,0,0,1
1,1,1.826816,-0.898713,0.719494,-1.046324,-1.521193,-0.749592,-0.582573,-0.161727,0,1,0
2,1,-0.547400,0.490011,0.351049,-0.887294,1.732315,-1.168931,-0.582573,-0.161727,1,0,0
3,0,-0.547400,0.490011,0.507980,-0.569234,0.253448,0.508425,-0.582573,-0.161727,0,1,0
4,0,-0.547400,-0.898713,-0.464987,-0.410204,0.844994,-2.846286,-0.582573,-0.161727,0,0,1


Treinamento

In [3]:
X = df.drop(columns=['IS_DELAYED'])
y = df['IS_DELAYED']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Distribuição das classes:\n{y.value_counts(normalize=True)}')
print(f'Treino: {X_train.shape}, Teste: {X_test.shape}')

Distribuição das classes:
IS_DELAYED
1    0.5
0    0.5
Name: proportion, dtype: float64
Treino: (1543796, 11), Teste: (385950, 11)


In [4]:
df_results, optimized_model = get_best_model(X_train, y_train, X_test, y_test)

Model              | Acc CV     | Acc Test   | F1 Test   
------------------------------------------------------------
XGBoost            | 0.7482     | 0.7480     | 0.7456    
RandomForest       | 0.7466     | 0.7458     | 0.7435    
GradientBoosting   | 0.7487     | 0.7479     | 0.7454    


In [4]:
best_model, best_params, results = tune_xgboost(X_train, y_train, X_test, y_test, n_iter=20)

Iniciando o ajuste de hiperparâmetros para o XGBoost.
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Melhores parâmetros encontrados:
subsample: 0.8
n_estimators: 1000
min_child_weight: 3
max_depth: 10
learning_rate: 0.01
gamma: 0
colsample_bytree: 0.6

Resultados:
Acc Test: 0.7482
F1 Test: 0.7457
Precisão: 0.7582
Recall: 0.7482
Best CV Score: 0.7463

Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.71      0.85      0.77    192975
           1       0.81      0.65      0.72    192975

    accuracy                           0.75    385950
   macro avg       0.76      0.75      0.75    385950
weighted avg       0.76      0.75      0.75    385950



In [ ]:
y_pred = best_model.predict(X_test)
plot_confusion_matrix(y_test, y_pred)

In [5]:
y_probs = best_model.predict_proba(X_test)[:, 1]
plot_roc_curve(y_test, y_probs)

**Que características aumentam a chance de atraso em um voo?**

In [6]:
df_importances = pd.DataFrame({
    'Variável': X.columns,
    'Importância': best_model.feature_importances_
}).sort_values(by='Importância', ascending=False)

plot_feature_importances(df_importances)